In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

## Prepare Data Set for Machine Learning

Load csv

In [5]:
df_raw = pd.read_csv(r"C:\Users\FrederikS\Documents\ECG_better\data\raw\French_Riviera_Multiple_campsites_datasets_capacity_over_200_anonymised.csv")

C:\Users\FrederikS\AppData\Local\Temp\ipykernel_51384\3222029601.py:1: DtypeWarning: Columns (19,22,23,24,25,26,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(r"C:\Users\FrederikS\Documents\ECG_better\data\raw\French_Riviera_Multiple_campsites_datasets_capacity_over_200_anonymised.csv")


Add additional columns, handle missing values and categorical columns

In [6]:
# add a Year column for easier time-based splitting later on

df_raw['Year'] = pd.to_datetime(df_raw['WeekStartDate']).dt.year

# extract the number of bedrooms from the "Bedrooms" column and convert it to an integer

df_raw["Bedrooms"] = df_raw["Bedrooms"].str.extract(r"(\d+)").astype(int)

# Split 'AccoTypeRangeCode' into 'AccommodationType' and 'RangeType'

df_raw[['AccommodationType', 'RangeType']] = (
    df_raw['AccoTypeRangeCode']
    .str.split(r'\s*\|\s*', n=1, expand=True)
)

# create a binary feature indicating whether the period is a special period or not

df_raw["is_special_period"] = df_raw["SpecialPeriodCode"].notna().astype(int)

Calculate the weighted average price, label the first price-data-point as 'initial price' and collapse the df so that only one row per ReservableOptionID remains

In [7]:

df_RO = df_raw.copy()

# Step 1: weighted value
df_RO['weighted_value'] = (
    df_RO['AverageDiscPriceForWeekBeforeArrival'] *
    df_RO['CumulativeHistoricalBookedNights']
)

# Step 2: aggregate at the correct level
agg = df_RO.groupby(
    ['ReservableOptionId', 'WeekBeforeArrival']
).agg(
    total_booked_nights=('CumulativeHistoricalBookedNights', 'sum'),   
    total_weighted_value=('weighted_value', 'sum')                     
).reset_index()

# Step 3: weighted average
agg['AverageWeightedPrice'] = (
    agg['total_weighted_value'] / agg['total_booked_nights']
).fillna(0)

# Step 4: merge back
df_RO = df_RO.merge(
    agg[['ReservableOptionId', 'WeekBeforeArrival',
         'AverageWeightedPrice', 'total_booked_nights']],
    on=['ReservableOptionId', 'WeekBeforeArrival'],
    how='left'
)

# Step 5: assign InitialPrice

df_RO = df_RO.sort_values(
    by=["ReservableOptionId", "WeekBeforeArrival"],
    ascending=[True, False]
)

df_RO["InitialPrice"] = (
    df_RO[df_RO["AverageWeightedPrice"] > 0]
    .groupby("ReservableOptionId")["AverageWeightedPrice"]
    .transform("first")
)
# Step 6: collapse to one row for each ReservableOptionId
df_RO = df_RO[df_RO['WeekBeforeArrival']==0].drop_duplicates(subset=['ReservableOptionId'])

# Step 7: drop rows where InitialPrice is NaN
df_RO = df_RO.dropna(subset=['InitialPrice'])

# Step 8: set index to Year
df_RO = df_RO.set_index('Year')

# Optional: keep same naming as before
#df_RO['OptionCumulativeBookedNights'] = df_RO['total_booked_nights']


## Train Test Split

In [8]:
# features and target variable

features = [
    'BrandGroupCode',
    'CampsiteCode',
    'is_special_period',
    'SeasonalCluster',
    'CampsiteType',
    'AccommodationType',
    'RangeType',
    'Airco',
    'Bedrooms',
    'DeckingType',
    'Bathrooms',
    'TV',
    'Capacity',
    
]
target = ['InitialPrice']

In [39]:
# Prepare training and test data
y_train = df_RO.loc[2024]['InitialPrice'].copy()
y_test = df_RO.loc[2025]['InitialPrice'].copy()

x_train = df_RO.loc[2024, features].copy()
x_test = df_RO.loc[2025, features].copy()

In [40]:
# Pool campsites with less than 30 unique options into "Other"

unique_counts = (
    df_RO.groupby("CampsiteCode")["ReservableOptionId"]
    .nunique()
    .reset_index(name="UniqueReservableOptions")
)

keep   = unique_counts[unique_counts["UniqueReservableOptions"] >= 30]["CampsiteCode"]                      

for frame in (x_train, x_test):
    frame["CampsiteCodeGrouped"] = frame["CampsiteCode"].where(
        frame["CampsiteCode"].isin(keep), "Other"
    )

x_train.drop(columns=["CampsiteCode"], inplace=True)
x_test.drop(columns=["CampsiteCode"], inplace=True)

In [41]:
# fill missing values in categorical columns with the most common value (mode)

x_train[["Airco", "TV", "DeckingType", "Bathrooms"]] = x_train[["Airco", "TV", "DeckingType","Bathrooms"]].apply(
    lambda col: col.fillna(col.mode()[0]))
x_test[["Airco", "TV", "DeckingType", "Bathrooms"]] = x_test[["Airco", "TV", "DeckingType","Bathrooms"]].apply(
    lambda col: col.fillna(col.mode()[0]))

C:\Users\FrederikS\AppData\Local\Temp\ipykernel_51384\4095183373.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda col: col.fillna(col.mode()[0]))
C:\Users\FrederikS\AppData\Local\Temp\ipykernel_51384\4095183373.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda col: col.fillna(col.mode()[0]))


## One-Hot-Encoding

In [42]:
# encode catergorical variables using one-hot encoding, ensuring the same columns in train and test sets

cols_to_encode = [
    "CampsiteCodeGrouped",
    "SeasonalCluster",
    "AccommodationType",
    "RangeType",
    "CampsiteType",
    "DeckingType",
    "BrandGroupCode"
]

x_train = pd.get_dummies(
    x_train,
    columns=cols_to_encode,
    drop_first=False  
)

x_test = pd.get_dummies(
    x_test,
    columns=cols_to_encode,
    drop_first=False  
)

x_test = x_test.reindex(columns=x_train.columns, fill_value=0)

## 2 Machine Learning Models

Ridge Regression

In [43]:
scaler = StandardScaler()

For Ridge Regression: Scaler

In [44]:
numeric_cols = ['Bedrooms', 'Bathrooms', 'Capacity']
x_train_scaled = x_train.copy()
x_train_scaled[numeric_cols] = scaler.fit_transform(x_train[numeric_cols])
x_test_scaled = x_test.copy()
x_test_scaled[numeric_cols] = scaler.transform(x_test[numeric_cols])

In [45]:
# 1) Fit Ridge, letting it choose the regularisation strength (alpha) by CV on TRAIN only.

alphas = np.logspace(-3, 3, 25)            # 0.001 ... 1000
ridge = RidgeCV(alphas=alphas)             
ridge.fit(x_train_scaled, y_train)
print(f"chosen alpha = {ridge.alpha_:.4g}")

# 2) Predict the held-out test season
y_pred = ridge.predict(x_test_scaled)

# 3) Evaluate — RMSE, MAE, R², MAPE 
def regression_report(y_true, y_pred, label=""):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{label:>12} | RMSE {rmse:8.2f} | MAE {mae:8.2f} | R² {r2:6.3f} | MAPE {mape:5.1f}%")
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}

ridge_metrics = regression_report(y_test, y_pred, "Ridge")

# 4) Interpretation — which features move the opening price, and which way
coef = pd.Series(ridge.coef_, index=x_train_scaled.columns).sort_values(key=np.abs, ascending=False)
print("\nTop 15 price drivers:")
print(coef.head(15).to_string())

chosen alpha = 1.778
       Ridge | RMSE    40.77 | MAE    29.78 | R²  0.787 | MAPE  34.0%

Top 15 price drivers:
SeasonalCluster_HS                 106.877258
CampsiteCodeGrouped_Campsite018     49.385153
SeasonalCluster_S2                 -41.204335
SeasonalCluster_WTR                -39.327667
CampsiteCodeGrouped_Campsite014    -35.805851
CampsiteCodeGrouped_Other          -31.911138
CampsiteCodeGrouped_Campsite006    -31.878168
RangeType_Budget                   -30.232835
SeasonalCluster_LS                 -29.697258
CampsiteCodeGrouped_Campsite020     27.178661
RangeType_Ultimate                  23.453043
CampsiteCodeGrouped_Campsite030     23.120698
CampsiteCodeGrouped_Campsite023    -20.991320
CampsiteCodeGrouped_Campsite028    -20.603431
CampsiteType_Partner                19.882765


Random Forest

In [46]:
# 1) Light CV search over the two knobs that control overfitting (parallel to RidgeCV's alpha).
param_grid = {
    "max_depth":        [None, 10, 20],   # how deep each tree may grow
    "min_samples_leaf": [1, 5, 10],       # min samples per leaf; higher = smoother, less overfit
}
rf_base = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
search = GridSearchCV(rf_base, param_grid, cv=5, scoring="neg_root_mean_squared_error")
search.fit(x_train, y_train)
rf = search.best_estimator_
print("best params:", search.best_params_)

# 2) Predict the held-out test season
y_pred_rf = rf.predict(x_test)

# 3) Evaluate — SAME function as Ridge, so the rows line up
rf_metrics = regression_report(y_test, y_pred_rf, "RandomForest")

# 4) Interpretation — feature importances (magnitude of reliance, NOT direction)
importances = pd.Series(rf.feature_importances_, index=x_train.columns).sort_values(ascending=False)
print("\nTop 15 features the forest relied on:")
print(importances.head(15).to_string())

best params: {'max_depth': None, 'min_samples_leaf': 10}
RandomForest | RMSE    39.06 | MAE    27.00 | R²  0.804 | MAPE  26.2%

Top 15 features the forest relied on:
SeasonalCluster_HS                 0.723090
RangeType_Budget                   0.103480
Capacity                           0.026909
TV                                 0.020724
CampsiteCodeGrouped_Campsite018    0.018182
Bedrooms                           0.016816
SeasonalCluster_S1                 0.016040
DeckingType_Small Decking          0.009306
RangeType_Comfort                  0.008943
DeckingType_Covered Decking        0.008267
RangeType_Ultimate                 0.007147
CampsiteCodeGrouped_Campsite030    0.007073
CampsiteCodeGrouped_Campsite017    0.006883
RangeType_Comfort+                 0.005302
CampsiteCodeGrouped_Campsite019    0.003203


Baseline

In [47]:
# --- Baselines: the bar the ML models must beat ---

# Baseline A — predict the global TRAIN mean price for every test option
baseline_global = np.full(len(y_test), y_train.mean())
regression_report(y_test, baseline_global, "Baseline-mean")

# Baseline B — predict the TRAIN mean price within a comparable segment
segment_cols = ["SeasonalCluster", "AccommodationType", "RangeType"]

seg_means   = df_RO.loc[2024].groupby(segment_cols)["InitialPrice"].mean()
global_mean = y_train.mean()

# map each 2025 option to its segment's 2024 mean; unseen segments -> global mean
baseline_segment = (
    df_RO.loc[2025]
    .set_index(segment_cols)
    .index.map(seg_means)
    .to_numpy(dtype="float64")
)
baseline_segment = np.where(np.isnan(baseline_segment), global_mean, baseline_segment)
regression_report(y_test, baseline_segment, "Baseline-segment")

Baseline-mean | RMSE    91.47 | MAE    79.60 | R² -0.074 | MAPE 106.5%
Baseline-segment | RMSE    41.87 | MAE    28.05 | R²  0.775 | MAPE  25.7%


{'rmse': 41.8734648183069,
 'mae': 28.052288540032723,
 'r2': 0.7749297556590955,
 'mape': np.float64(25.696646731480122)}

Per Seasonalcluster Evaluation

In [48]:

per_cluster = pd.DataFrame({
    "SeasonalCluster": df_RO.loc[2025, "SeasonalCluster"].to_numpy(),
    "y_true":          y_test.to_numpy(),
    "Baseline-segment": baseline_segment,
    "Ridge":            y_pred,
    "RandomForest":     y_pred_rf,
})

rmse_by_cluster = (
    per_cluster
    .groupby("SeasonalCluster")
    .apply(lambda g: pd.Series({
        "n":                len(g),
        "Baseline-segment": mean_squared_error(g["y_true"], g["Baseline-segment"]) ** 0.5,
        "Ridge":            mean_squared_error(g["y_true"], g["Ridge"]) ** 0.5,
        "RandomForest":     mean_squared_error(g["y_true"], g["RandomForest"]) ** 0.5,
    }), include_groups=False)
    .round(2)
)

print("RMSE per SeasonalCluster (lower = better):")
print(rmse_by_cluster.to_string())

# RF improvement over the segment baseline, per cluster
rmse_by_cluster["RF_vs_Baseline_%"] = (
    (rmse_by_cluster["RandomForest"] - rmse_by_cluster["Baseline-segment"])
    / rmse_by_cluster["Baseline-segment"] * 100
).round(1)
print("\nNegative % = RandomForest beats the baseline in that tier:")
print(rmse_by_cluster["RF_vs_Baseline_%"].to_string())

RMSE per SeasonalCluster (lower = better):
                     n  Baseline-segment  Ridge  RandomForest
SeasonalCluster                                              
HS               556.0             61.81  54.65         56.97
LS               600.0             24.25  32.73         23.31
S1               127.0             36.62  26.20         35.20
S2               269.0             23.87  31.22         23.05
WTR               54.0             16.91  17.47         18.97

Negative % = RandomForest beats the baseline in that tier:
SeasonalCluster
HS     -7.8
LS     -3.9
S1     -3.9
S2     -3.4
WTR    12.2


No single model wins everywhere. RF is the best all-round choice (beats baseline in 4/5 tiers), but Ridge is materially better in the two important tiers (HS, S1)